# Task 2 :  Sentiment Analysis using NLP Pipeline & ML Models

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('all_kindle_review.csv')

# 1. Data Understanding

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df.shape

(12000, 11)

In [5]:
df.isnull().sum()

Unnamed: 0.1       0
Unnamed: 0         0
asin               0
helpful            0
rating             0
reviewText         0
reviewTime         0
reviewerID         0
reviewerName      38
summary            2
unixReviewTime     0
dtype: int64

### 1. Keeping important columns only

In [6]:
df = df[['reviewText','rating']]

In [7]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


### 2. Building core logic
## If rating is less than 3 = Negative Sentiment
## If rating is greater than 3 = Positive Sentiment

In [8]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [9]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [10]:
df['rating'] = df['rating'].apply(lambda x : 0 if x<3 else 1)

In [11]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

# 2. NLP Preprocessing

In [12]:
import re
import string
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load stopwords once
STOPWORDS = set(stopwords.words('english')) - {"no", "not" , "never"}

def clean_text(text):
    # 1. Convert to string (safety)
    text = str(text)
    
    # 2. Remove HTML tags
    text = BeautifulSoup(text, "lxml").get_text()
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+|ftp\S+|ssh\S+', '', text)
    
    # 4. Remove special characters (keep alphabets + numbers)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 5. Convert to lowercase
    text = text.lower()
    
    # 6. Remove numbers (optional → comment if needed)
    text = ''.join([char for char in text if not char.isdigit()])
    
    # 7. Handle repeated characters (e.g., gooooood → good)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # 8. Remove extra spaces
    text = " ".join(text.split())
    
    # 9. Tokenization
    tokens = word_tokenize(text)
    
    # 10. Remove stopwords but KEEP "no", "not"
    tokens = [word for word in tokens if word not in STOPWORDS]
    
    # 11. Remove very short words (≤2), keep "no", "not"
    tokens = [word for word in tokens if len(word) > 2 or word in ["no", "not"]]
    
    # 12. Join back
    return " ".join(tokens)

In [13]:
df['reviewText'] = df['reviewText'].apply(clean_text)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1156\1691632688.py:15: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "lxml").get_text()


In [14]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,not expect type book library pleased find pric...,1


### Applying Lemmatization for good accuracy

In [15]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [16]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [17]:
df['reviewText'] = df['reviewText'].apply(lambda x : lemmatize_words(x))

In [18]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,not expect type book library pleased find pric...,1


# Steps
### 1. Preprocessing and cleaning - Done
### 2. Train Test Split - First performing train test split inorder to avoid Data Leakage Problem
### 3. Apply : BOW , TFIDF , Word2Vec , Avg Word2Vec
### 4. Training the data on various ML models
### 5. Model Evaluation

In [19]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.2)

# 3. Feature Engineering

## 1. BOW (Bag Of Words)

In [20]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

## 2. TF-IDF (Term Frequency - Inverse Document Frequency)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

## 3. Word 2 Vec

In [41]:
X_train_tokens = X_train.apply(lambda x: x.split())
X_test_tokens = X_test.apply(lambda x: x.split())

In [43]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

## 4. Average Word2Vec

In [47]:
import numpy as np

def avg_word2vec(sentence, model, vector_size):
    vectors = []
    
    for word in sentence:
        if word in model.wv:
            vectors.append(model.wv[word])
    
    # Handle empty case
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    return np.mean(vectors, axis=0)

In [49]:
X_train_w2v = np.array([
    avg_word2vec(words, w2v_model, 100)
    for words in X_train_tokens
])

X_test_w2v = np.array([
    avg_word2vec(words, w2v_model, 100)
    for words in X_test_tokens
])

# 4. Model Building

## 1. Logistic Regression

### 1.1 Logistic Regression Model on BOW

In [70]:
from sklearn.linear_model import LogisticRegression

lr_model_bow = LogisticRegression()
lr_model_bow.fit(X_train_bow, y_train)

y_pred_lr_bow = lr_model_bow.predict(X_test_bow)

### 1.2 Logistic Regression Model on TF-IDF

In [72]:
lr_model_tfidf = LogisticRegression()
lr_model_tfidf.fit(X_train_tfidf,y_train)
y_pred_lr_tfidf = lr_model_tfidf.predict(X_test_tfidf)

### 1.3 Logistic Regression Model on Word2Vec and Avg Word2Vec

In [79]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_w2v = scaler.fit_transform(X_train_w2v)
X_test_w2v = scaler.transform(X_test_w2v)

In [80]:
lr_model_w2v = LogisticRegression(max_iter=1000,solver='lbfgs')
lr_model_w2v.fit(X_train_w2v,y_train)
y_pred_lr_w2v = lr_model_w2v.predict(X_test_w2v)

## 2. Naive Bayes

### 2.1 Naive Bayes Model on BOW - MultinomialNB best choice

In [87]:
from sklearn.naive_bayes import MultinomialNB
nb_model_bow = MultinomialNB()
nb_model_bow.fit(X_train_bow,y_train)
y_pred_nb_bow = nb_model_bow.predict(X_test_bow)

### 2.2 Naive Bayes Model on TFIDF - MultinomialNB best choice

In [90]:
nb_model_tfidf = MultinomialNB()
nb_model_tfidf.fit(X_train_tfidf,y_train)
y_pred_nb_tfidf = nb_model_tfidf.predict(X_test_tfidf)

### 2.3 Naive Bayes Model on TFIDF - GaussianNB best choice

In [97]:
from sklearn.naive_bayes import GaussianNB
nb_model_w2v = GaussianNB()
nb_model_w2v.fit(X_train_w2v,y_train)
y_pred_nb_w2v = nb_model_w2v.predict(X_test_w2v)

## 3. Decision Tree

### 3.1 Decision Tree Model on BOW

In [102]:
from sklearn.tree import DecisionTreeClassifier

dt_model_bow = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    criterion='gini',
    random_state=42
)
dt_model_bow.fit(X_train_bow, y_train)
y_pred_dt_bow = dt_model_bow.predict(X_test_bow)

### 3.2 Decision Tree Model on TFIDF

In [104]:
dt_model_tfidf = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    criterion='gini',
    random_state=42
)
dt_model_tfidf.fit(X_train_tfidf, y_train)
y_pred_dt_tfidf = dt_model_tfidf.predict(X_test_tfidf)

### 3.3 Decision Tree Model on TFIDF

In [114]:
dt_model_w2v = DecisionTreeClassifier(
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    criterion='entropy',
    random_state=42
)
dt_model_w2v.fit(X_train_w2v, y_train)
y_pred_dt_w2v = dt_model_w2v.predict(X_test_w2v)

## 4. Random Forest

### 4.1 Random Forest Model on BOW

In [122]:
from sklearn.ensemble import RandomForestClassifier

rf_model_bow = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model_bow.fit(X_train_bow, y_train)
y_pred_rf_bow = rf_model_bow.predict(X_test_bow)

### 4.2 Random Forest Model on TFIDF

In [125]:
rf_model_tfidf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model_tfidf.fit(X_train_tfidf, y_train)
y_pred_rf_tfidf = rf_model_tfidf.predict(X_test_tfidf)

### 4.3 Random Forest Model on Word2Vec

In [127]:
rf_model_w2v = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model_w2v.fit(X_train_w2v, y_train)
y_pred_rf_w2v = rf_model_w2v.predict(X_test_w2v)

## 5. XGBOOST

## 5.1 XGBoost Model on BOW

In [148]:
from xgboost import XGBClassifier

xgb_model_bow = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_model_bow.fit(X_train_bow, y_train)
y_pred_xgb_bow = xgb_model_bow.predict(X_test_bow)

## 5.2 XGBoost Model on TFIDF

In [150]:
xgb_model_tfidf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_model_tfidf.fit(X_train_tfidf, y_train)
y_pred_xgb_tfidf = xgb_model_tfidf.predict(X_test_tfidf)

## 5.3 XGBoost Model on Word2Vec

In [152]:
xgb_model_w2v = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_model_w2v.fit(X_train_w2v, y_train)
y_pred_xgb_w2v = xgb_model_w2v.predict(X_test_w2v)

# 5. Model Evaluation

In [153]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(model_name, y_test):
    
    # Mapping model names to predictions
    predictions = {
        "nb_bow": y_pred_nb_bow,
        "nb_tfidf": y_pred_nb_tfidf,
        "nb_w2v": y_pred_nb_w2v,
        
        "dt_bow": y_pred_dt_bow,
        "dt_tfidf": y_pred_dt_tfidf,
        "dt_w2v": y_pred_dt_w2v,
        
        "rf_bow": y_pred_rf_bow,
        "rf_tfidf": y_pred_rf_tfidf,
        "rf_w2v": y_pred_rf_w2v,
        
        "xgb_bow": y_pred_xgb_bow,
        "xgb_tfidf": y_pred_xgb_tfidf,
        "xgb_w2v": y_pred_xgb_w2v
    }
    
    y_pred = predictions.get(model_name)
    
    if y_pred is None:
        print("Invalid model name!")
        return
    
    print(f"\nEvaluation for: {model_name.upper()}")
    print("-" * 40)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [164]:
import pandas as pd

def compare_all_models(y_test):
    
    models = [
        "nb_bow", "nb_tfidf", "nb_w2v",
        "dt_bow", "dt_tfidf", "dt_w2v",
        "rf_bow", "rf_tfidf", "rf_w2v",
        "xgb_bow", "xgb_tfidf", "xgb_w2v"
    ]
    
    results = []
    
    for model in models:
        predictions = {
            "nb_bow": y_pred_nb_bow,
            "nb_tfidf": y_pred_nb_tfidf,
            "nb_w2v": y_pred_nb_w2v,
            "dt_bow": y_pred_dt_bow,
            "dt_tfidf": y_pred_dt_tfidf,
            "dt_w2v": y_pred_dt_w2v,
            "rf_bow": y_pred_rf_bow,
            "rf_tfidf": y_pred_rf_tfidf,
            "rf_w2v": y_pred_rf_w2v,
            "xgb_bow": y_pred_xgb_bow,
            "xgb_tfidf": y_pred_xgb_tfidf,
            "xgb_w2v": y_pred_xgb_w2v
        }
        
        y_pred = predictions[model]
        
        results.append({
            "Model": model,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred)
        })
    
    df_results = pd.DataFrame(results).sort_values(by="Accuracy")
    return df_results

In [166]:
compare_all_models(y_test)

,Model,Accuracy,Precision,Recall,F1 Score
6,rf_bow,0.677500,0.676557,1.000000,0.807079
7,rf_tfidf,0.677917,0.676987,0.999382,0.807184
1,nb_tfidf,0.703750,0.695185,0.998765,0.819772
5,dt_w2v,0.716250,0.783898,0.799876,0.791807
2,nb_w2v,0.717500,0.862760,0.691167,0.767490
3,dt_bow,0.728750,0.759657,0.874614,0.813092
4,dt_tfidf,0.730833,0.746579,0.909821,0.820156
8,rf_w2v,0.777500,0.805290,0.883879,0.842756
11,xgb_w2v,0.790417,0.821429,0.880791,0.850075
9,xgb_bow,0.815417,0.836000,0.903644,0.868507


In [184]:
evaluate_model('nb_bow',y_test) # similar we can perform on all model types


Evaluation for: NB_BOW
----------------------------------------
Accuracy  : 0.8392
Precision : 0.8624
Recall    : 0.9061
F1 Score  : 0.8837

Confusion Matrix:
[[ 547  234]
 [ 152 1467]]


## Model Evaluation based on random user input

In [203]:
import re

# 1. Simple preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text


# 2. Prediction function
def predict_sentiment(text):
    text = preprocess_text(text)
    
    # convert using your trained BoW vectorizer
    vector = bow.transform([text])
    
    prediction = nb_model_bow.predict(vector)[0]
    
    if prediction == 1:
        return "Positive 😊"
    else:
        return "Negative 😞"


# 3. User input loop
while True:
    user_input = input("Enter a review (or type 'exit'): ")
    
    if user_input.lower() == 'exit':
        break
    
    result = predict_sentiment(user_input)
    print("Sentiment:", result)
    print("-" * 50)

Enter a review (or type 'exit'):  The overall quality of the product is good 


Sentiment: Positive 😊
--------------------------------------------------


Enter a review (or type 'exit'):  Ohh The product which i purchased is not the one i got here


Sentiment: Negative 😞
--------------------------------------------------


Enter a review (or type 'exit'):  what I think is that this is not appropriate product to buy


...

Enter a review (or type 'exit'):  worst product ever used


Sentiment: Negative 😞
--------------------------------------------------


Enter a review (or type 'exit'):  looking forward to buy such beautiful products


Sentiment: Positive 😊
--------------------------------------------------


Enter a review (or type 'exit'):  exit


# 6. Comparison & Insights

## ✅ Best Preprocessing
- Cleaning + lowercasing + removing noise worked well
- Keeping important words like "not" helped sentiment detection
- Stopword removal worked best for TF-IDF and BoW

---

## ✅ Best Vectorization
- TF-IDF performed best overall
- It balances importance of words better than BoW
- Word2Vec gave good semantic understanding but slightly lower accuracy

---

## ✅ Best Model
- Naive Bayes (BoW) → Highest Accuracy: **0.8391**
- XGBoost (TF-IDF) → Strong overall performance: **0.8220**
- Random Forest (Word2Vec) → Best among Word2Vec: **0.7775**

👉 Final Best Combination:
**BoW + MultinomialNB**

---

## ⚖️ Trade-offs

### BoW + Naive Bayes
✔ Highest accuracy  
✔ Simple and fast  
❌ Ignores context (no semantics)

---

### TF-IDF + XGBoost
✔ Balanced performance (accuracy + precision + recall)  
✔ Better feature importance handling  
❌ Slightly slower than Naive Bayes  

---

### Word2Vec + Tree Models
✔ Captures semantic meaning  
✔ Lower dimensional (efficient)  
❌ Slightly lower accuracy in this dataset  

---

## 🎯 Final Insight
- Simpler models (Naive Bayes + BoW) performed best on this dataset
- Advanced models (XGBoost, Word2Vec) are more powerful but need more tuning
- Choice depends on:
  - Accuracy → BoW + NB
  - Balance & robustness → TF-IDF + XGBoost
  - Semantic understanding → Word2Vec models